# 回归任务 Demo

本 Notebook 展示了使用不同机器学习算法进行回归任务的方法。

## 数据集
- California Housing Dataset

In [ ]:
import sys
sys.path.append('..')

from src.data_loader import DataLoader
from src.evaluation import evaluate_regression, print_metrics
from src.models.regression import REGRESSION_MODELS
import time

In [ ]:
# 加载数据
data_loader = DataLoader()
X_train, X_test, y_train, y_test = data_loader.load_california_housing()
print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")

In [ ]:
# 运行所有回归算法
results = {}
for algo_key, model_class in REGRESSION_MODELS.items():
    print(f"\n{'-'*40}")
    print(f"Training: {algo_key}")
    print(f"{'-'*40}")
    
    start_time = time.time()
    model = model_class()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    train_time = time.time() - start_time
    
    metrics = evaluate_regression(y_test, y_pred)
    metrics["train_time"] = train_time
    
    print_metrics("regression", metrics)
    print(f"  Training Time: {train_time:.2f}s")
    
    results[algo_key] = metrics

In [ ]:
# AutoGluon Baseline
try:
    from autogluon.tabular import TabularPredictor
    
    print("Running AutoGluon Baseline...")
    
    train_df = X_train.copy()
    train_df['target'] = y_train.values
    
    start_time = time.time()
    predictor = TabularPredictor(
        label='target',
        problem_type='regression',
        eval_metric='rmse'
    )
    predictor.fit(train_df, time_limit=120)
    train_time = time.time() - start_time
    
    y_pred = predictor.predict(X_test)
    
    metrics = evaluate_regression(y_test, y_pred)
    metrics["train_time"] = train_time
    
    print("\nAutoGluon Baseline Results:")
    print_metrics("regression", metrics)
    print(f"  Training Time: {train_time:.2f}s")
    
    results["autogluon"] = metrics
    
except ImportError:
    print("AutoGluon not installed. Skipping.")
except Exception as e:
    print(f"AutoGluon error: {e}")

In [ ]:
# 结果汇总
import pandas as pd

df_results = pd.DataFrame(results).T
df_results = df_results.round(4)
print("\n" + "="*60)
print("回归任务结果汇总")
print("="*60)
print(df_results)